<a href="https://colab.research.google.com/github/Kusinga321/Nvidia-interview/blob/main/even_easier_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!nvidia-smi

Thu Oct 22 21:35:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile test.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CHECK(call) do {                                              \
    cudaError_t err = (call);                                         \
    if (err != cudaSuccess) {                                         \
        printf("CUDA error: %s (line %d)\n",                          \
               cudaGetErrorString(err), __LINE__);                    \
        exit(1);                                                      \
    }                                                                 \
} while (0)

__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    // GPU info
    int dev;
    cudaDeviceProp prop;
    CHECK(cudaGetDevice(&dev));
    CHECK(cudaGetDeviceProperties(&prop, dev));
    printf("GPU: %s (compute capability %d.%d)\n",
           prop.name, prop.major, prop.minor);

    const int n = 1 << 20;  // ~1 million elements
    size_t bytes = n * sizeof(float);

    // Host memory
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) { h_a[i] = i; h_b[i] = 2.0f * i; }

    // Device memory
    float *d_a, *d_b, *d_c;
    CHECK(cudaMalloc(&d_a, bytes));
    CHECK(cudaMalloc(&d_b, bytes));
    CHECK(cudaMalloc(&d_c, bytes));
    CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    // Launch kernel
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Copy back and verify
    CHECK(cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    for (int i = 0; i < n; i++) {
        if (fabsf(h_c[i] - 3.0f * i) > 1e-3f * fmaxf(1.0f, 3.0f * i)) errors++;
    }

    if (errors == 0) printf("SUCCESS: all %d results are correct!\n", n);
    else             printf("FAILED: %d wrong results\n", errors);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return errors ? 1 : 0;
}

Writing test.cu


In [ ]:
!nvcc -arch=sm_75 test.cu -o test && ./test

GPU: Tesla T4 (compute capability 7.5)
SUCCESS: all 1048576 results are correct!


In [67]:
%%writefile add.cu

#include <iostream>
#include <math.h>

// Kernel function to add the elements of two arrays
__global__
void add(int n, float *x, float *y)
{
  int idx = threadIdx.x + blockIdx.x * blockDim.x;
 for (int i = idx; i < n; i+=blockDim.x * gridDim.x)
   y[i] = x[i] + y[i];

  printf("%d %d %d\n", threadIdx.x, threadIdx.y, threadIdx.z);
}

int main(void)
{
 int N = 1<<8;
 float *x, *y;

 // Allocate Unified Memory accessible from CPU or GPU
 cudaMallocManaged(&x, N*sizeof(float));
 cudaMallocManaged(&y, N*sizeof(float));

 // initialize x and y arrays on the host
 for (int i = 0; i < N; i++) {
   x[i] = 1.0f;
   y[i] = 2.0f;
 }

cudaMemLocation loc;
loc.type = cudaMemLocationTypeDevice;
loc.id = 0;                       // GPU nr 0
cudaMemPrefetchAsync(x, N*sizeof(float), loc, 0, 0);   // (ptr, rozmiar, gdzie, flags, stream)
cudaMemPrefetchAsync(y, N*sizeof(float), loc, 0, 0);

 // Run kernel on 1M elements on the GPU
 int blockSize = 256;
int numBlocks = (N + blockSize - 1) / blockSize;
add<<<numBlocks, blockSize>>>(N, x, y);

 // Wait for GPU to finish before accessing on host
 cudaDeviceSynchronize();

 // Check for errors (all values should be 3.0f)
 float maxError = 0.0f;
 for (int i = 0; i < N; i++) {
   maxError = fmax(maxError, fabs(y[i]-3.0f));
 }
 std::cout << "Max error: " << maxError << std::endl;

 // Free memory
 cudaFree(x);
 cudaFree(y);
  return 0;
}

Overwriting add.cu


In [70]:
!nvcc -arch=sm_75 add.cu -o add_cuda && ./add_cuda

224 0 0
225 0 0
226 0 0
227 0 0
228 0 0
229 0 0
230 0 0
231 0 0
232 0 0
233 0 0
234 0 0
235 0 0
236 0 0
237 0 0
238 0 0
239 0 0
240 0 0
241 0 0
242 0 0
243 0 0
244 0 0
245 0 0
246 0 0
247 0 0
248 0 0
249 0 0
250 0 0
251 0 0
252 0 0
253 0 0
254 0 0
255 0 0
128 0 0
129 0 0
130 0 0
131 0 0
132 0 0
133 0 0
134 0 0
135 0 0
136 0 0
137 0 0
138 0 0
139 0 0
140 0 0
141 0 0
142 0 0
143 0 0
144 0 0
145 0 0
146 0 0
147 0 0
148 0 0
149 0 0
150 0 0
151 0 0
152 0 0
153 0 0
154 0 0
155 0 0
156 0 0
157 0 0
158 0 0
159 0 0
96 0 0
97 0 0
98 0 0
99 0 0
100 0 0
101 0 0
102 0 0
103 0 0
104 0 0
105 0 0
106 0 0
107 0 0
108 0 0
109 0 0
110 0 0
111 0 0
112 0 0
113 0 0
114 0 0
115 0 0
116 0 0
117 0 0
118 0 0
119 0 0
120 0 0
121 0 0
122 0 0
123 0 0
124 0 0
125 0 0
126 0 0
127 0 0
0 0 0
1 0 0
2 0 0
3 0 0
4 0 0
5 0 0
6 0 0
7 0 0
8 0 0
9 0 0
10 0 0
11 0 0
12 0 0
13 0 0
14 0 0
15 0 0
16 0 0
17 0 0
18 0 0
19 0 0
20 0 0
21 0 0
22 0 0
23 0 0
24 0 0
25 0 0
26 0 0
27 0 0
28 0 0
29 0 0
30 0 0
31 0 0
32 0 0
33 0 0
34 0 0
3

In [ ]:
!nsys profile --stats=true ./add_cuda

Max error: 0
Generating '/tmp/nsys-report-dd5f.qdstrm'
[1/8] [========================100%] report8.nsys-rep
[2/8] [========================100%] report8.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/report8.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  ----------------------
     77.9      563,542,454         23  24,501,845.8  10,079,099.0     1,771  292,519,962  60,582,015.7  poll                  
     21.3      154,417,907        506     305,173.7      12,626.0     1,043   16,841,027     960,891.1  ioctl                 
      0.2        1,733,898         29      59,789.6      10,274.0     5,848    1,215,563     222,876.7  mmap64                
      0.2        1,575,079         10    